#  Performance study: NSR vs. V magnitude

This notebook shows how Fig. 17 was generated from the simulated dataset.

### Setup notebook

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt

# PlatoSim
import platosim.plot            as pt
import platosim.utilities       as ut
import platosim.referenceFrames as rf
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

### Functions

In [ ]:
# Function to correctly set the input magnitude of star
def add_columns(cfile, df):
    # Load input catalogue
    dc = pd.read_feather(cfile)
    N = len(dc)
    # Merge the two data frames
    mag  = np.array([])
    ncon = np.array([])
    Teff = np.array([])
    for i in range(N):
        nobs = len(df[df.star == i+1])
        mags = dc.mag.iloc[i] * np.ones(nobs)
        mag  = np.concatenate((mag, mags))
        ncons = dc.ncon.iloc[i] * np.ones(nobs)
        ncon  = np.concatenate((ncon, ncons))
        Teffs = dc.Teff.iloc[i] * np.ones(nobs)
        Teff  = np.concatenate((Teff, Teffs))
    df["mag"]  = mag
    df["ncon"] = ncon
    df["Teff"] = Teff
    # Correct columns of magnitudes
    df['mag'] = mag[:len(df)]
    return df

In [ ]:
def findNSR(df):
    
    # Show methodology behind the NSR calculation used
    time = df.time/86400.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

    # Bin to devide data                                                                                                                                                                                
    dt = 1/24.                                                                                                                                                                                          
    nbins = round( (time.max() - time.min()) / dt) + 1                                                                                                                                      
    tbins = np.linspace(time.min(), time.max(), nbins)
    nbin  = len(df[time.between(tbins[0], tbins[1])])                                                                                                                                             

    # Bin data                                                                                                                                                                                          
    flux_dex = df.columns.get_loc('flux')                                                                                                                                                               
    data  = [df[time.between(tbins[i], tbins[i+1])].to_numpy() for i in range(nbins-1)]                                                                                                           
    sigma = np.array([data[i][:,flux_dex].std() for i in range(len(data))])                                                                                                                                                                                                                                                                                                                                     

    # Return NSR [ppm]                                                                                                                                                                                       
    NSR = np.mean(sigma) / np.sqrt(nbin) 
    return NSR

### Download data from KUL FTP server

In [ ]:
# User parameters
idir = "/lhome/nicholas/data/platosimPaper/NSR_old"

In [ ]:
# Load stellar catalogue used
vfile = idir + "/starcat_all_SPF_CamVis24_NewCat_targets.ftr"
dv = pd.read_feather(vfile)
dv.head()

## Test example: NSR for a single star

In [ ]:
# Load light curve object for the first star only and unpack the data
lcs = LightCurve(f"{idir}/000000001", mode="multi")
lcs.unpack()

In [ ]:
# Fetch all files
filenames = lcs.files("hdf5")

In [ ]:
# Check data output for a single ligth curve
lc = LightCurve(filenames[0])
lc.data().head()

In [ ]:
# Merge data across cameras [flux -> ppm]
lc, ncam, flag = lcs.merge(quarter=1, flux_group_mean=True, suffix='hdf5')

In [ ]:
# Show the first light curve an binned data for which NSR is estimated from
fig, ax = lc.plot(time_unit="h", binsize=1, alpha=0.5, figsize=(9,5));
ax.set_ylabel('Flux [ppm]');

In [ ]:
# Calculate the NSR
lc.getNSR(influx='ppm')

## Analysis of **Single** N-CAM LCs

In [ ]:
idir  = "/lhome/nicholas/data/platosimPaper/NSR_old" 
ofile = idir + "/results_per_camera_old.ftr"

In [ ]:
# lcs = LightCurve(idir, mode="multi")
# lcs.run_NSRvsMag_analysis_perCamera(ofile, 10000, suffix="hdf5")

In [ ]:
# Load results and sort logically
df = pd.read_feather(ofile)

# Add stellar magnitudes to table
df = add_columns(vfile, df)

# Order after contaminants and reverse to show 0 contaminants on top
df = df.sort_values(by=['ncon'])

In [ ]:
# Remove 4 stars with more than 20 contaminants 
# (simply to better visualise the the bulk of the star with contaminant)
df = df[df.ncon < 20]
df.head()

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, column="ncon", passband='V', residuals="multi", 
                                show_ncam_noise_limits=1, show_saturation_limits=True,
                                show_targets=True, legend=True, grid=False, figsize=(9.2,6.2))
# Settings
ax.set_ylim(5, 1000)
ax.set_title('a) Noise budget at camera level for BOL')
# Save figure
# fig.savefig('nsrCameraLevel.png', bbox_inches='tight', dpi=300);

### Test influence of saturation

This simulation study has been using:

- $g_{\rm FEE}=0.0222$ e-/muV 
- $g_{\rm CCD}=2.21$ ADU/muV
- $1/g = 1/(g_{\rm FEE} \times g_{\rm FEE}) = 0.38$ ADU/e- 

So saturation kicks in at $F_{\rm max} = 900,000$ e- $/ g = 44,156$ ADU. Bias is 1000 ADU and sky backround would be around (60 e- $\times$ 21 s) / $g = 479$ ADU. We observe $F_{\rm max} = 43,756$ ADU below which is higher than expected.

In [ ]:
# Magnitude of star
m = 8

# Initialise PlatSim (we overwrite each output file)
sim = Simulation("output_jitter")

# Simulate 1 day
sim["ObservingParameters/NumExposures"] = 1

# Change reference flux
sim['ObservingParameters/Fluxm0'] = 0.7324478224428527e8
    
# sim['CCD/Gain/RefValueLeft']  = 3.0   
# sim['CCD/Gain/RefValueRight'] = 3.0
    
# Set the subfield
sim["SubField/ZeroPointRow"]    = 1000
sim["SubField/ZeroPointColumn"] = 1000
colStar = sim["SubField/NumColumns"] = 6
rowStar = sim["SubField/NumRows"]    = 13

# Define catalogue
row = np.array([int(rowStar/2)+0.5]) + sim["SubField/ZeroPointRow"]
col = np.array([int(rowStar/2)+0.5]) + sim["SubField/ZeroPointColumn"]
mag = np.array([m])
ID  = [1]

# Create star catalogue file
starcatFile = f'{os.getcwd()}/starcat.txt'
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)

# Create photometry file
photometryFile = f'{os.getcwd()}/photometry.txt'
sim.createPhotometryTargetFile(ID, photometryFile)

# Run the simulation
f = sim.run(removeOutputFile=True, executionTime=True)

# im = f.getImage(0)
# f.showMap(im, figsize=(6,6));
fig, ax = f.showImage(clipPercentile=5, imgScale="clip", fontSize=20, figsize=(6,6), 
                      colorMap="cubehelix", colorBar=True, showGrid=True,
                      showStarPositions='PIC', showMaskOfStarID=1);

# Print max pixel value
np.max(f.getImage(0))

### Test jitter noise regime

In [ ]:
path = "/lhome/nicholas/Nextcloud/Platoman/Models/Jitter/Prime2020jan"
df_jitter = pd.read_csv(f'{path}/01_PLATO_PDR_FPM_02_longrun_APE.csv', delimiter=';', 
                     names=['t', 'x', 'y', 'z'], skiprows=[0,1])

# Let time series start at zero seconds
df_jitter['t'] -= df_jitter['t'].iloc[0]

# Convert angles from rad -> arcsec
for n in ['x', 'y', 'z']:
    df_jitter[n] = df_jitter[n] * 206265. 
    
df_jitter.head()

In [ ]:
# Plot jitter time series]
pt.plotYawPitchRollTimeSeries(df_jitter.t/3600, np.array([df_jitter.x, df_jitter.y, df_jitter.z]), units=["hours", "arcsec"]);

In [ ]:
# Determine PSD of yaw
from scipy.signal import periodogram
from scipy.ndimage import median_filter
f, psd = periodogram(df_jitter["x"], 8, scaling='density')
f   *= 1e3  # [mHz]
psd *= 1e6  # [ppm^2/muHz]
psd_med = median_filter(psd, 100)

In [ ]:
# Plot PSD of yaw
plt.figure(figsize=(8,4))
plt.plot(f, psd)
plt.plot(f, psd_med, 'r-')
plt.xlabel(r"Frequency, $\nu$ [mHz]")
plt.ylabel(r"PSD [ppm mHz$^{-1}$]")
plt.xlim(1e0, f.max())
plt.ylim(1e-6, 1e6)
plt.xscale('log')
plt.yscale('log')
plt.tight_layout();

In [ ]:
# Funtion to calculate the NSR for different jitter realisations
def nsr(amplitude, magnitude, model):

    # Initialise PlatSim (we overwrite each output file)
    sim = Simulation("output_jitter")

    # Don't save anything to output to save time
    sim.turnOffAllOutput()

    # Simulate 1 day
    sim["ObservingParameters/NumExposures"] = int(86400/25.) 

    # Change reference flux
    sim['ObservingParameters/Fluxm0'] = 0.7324478224428527e8
    
    # Set higher gain values -> 55 kADU for saturation
#     sim['CCD/Gain/RefValueLeft']  = 3.0   
#     sim['CCD/Gain/RefValueRight'] = 3.0
    
    # Set jitter file
    if model == 'FromFile':
        jitterFileName = f'{os.getcwd()}/AOCS_amplitude{a}.txt'
        sim["Platform/UseJitter"]      = 'yes'
        sim["Platform/JitterSource"]   = 'FromFile'
        sim["Platform/JitterFileName"] = jitterFileName
    else:
        sim["Platform/JitterSource"]   = 'FromRedNoise'
        sim["Platform/JitterYawRms"]   = 0.04 * a
        sim["Platform/JitterPitchRms"] = 0.04 * a
        sim["Platform/JitterRollRms"]  = 0.04 * a
    
    # Set the subfield
    sim["SubField/ZeroPointRow"]    = 1000
    sim["SubField/ZeroPointColumn"] = 1000
    sim["SubField/NumColumns"]      = 6
    sim["SubField/NumRows"]         = 6

    # Define catalogue
    row = np.array([4.0]) + sim["SubField/ZeroPointRow"]
    col = np.array([4.0]) + sim["SubField/ZeroPointColumn"]
    mag = np.array([magnitude])
    ID  = [1]

    # Create star catalogue file
    starcatFile = f'{os.getcwd()}/starcat.txt'
    sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)

    # Create photometry file
    photometryFile = f'{os.getcwd()}/photometry.txt'
    sim.createPhotometryTargetFile(ID, photometryFile)

    # Run the simulation
    f = sim.run(removeOutputFile=True, executionTime=True)
    
    # Fetch the photometry
    df = f.getLightCurve(ID)
    lc = LightCurve(df, 'multi')
    
    # Calculate NSR in 1h
    return lc.getNSR()

In [ ]:
##### Generate array of magnitudes
model = 'FromFile'
colors = ['c', 'm', 'y', 'r']
amplitudes = [25, 50, 75, 100]
magnitudes = [8] #np.arange(7.2, 8.6, 0.1)
cols = np.append('mag', [f'jitter{amplitudes[i]}' for i in range(len(amplitudes))])
nd = np.zeros((len(amplitudes)+1, len(magnitudes))).T
df0 = pd.DataFrame(nd, columns=cols)
df0

In [ ]:
# Check amplitudes
a = 10
df0_jitter = df_jitter.copy()
df0_jitter.x *= a
df0_jitter.y *= a
df0_jitter.z *= a
df0_jitter

In [ ]:
# Extract 
for a in amplitudes:
    if model == 'FromFile':
        # Create a AOCS file withSplit time from signals to adapt to plots below
        df0_jitter = df_jitter.copy()
        df0_jitter.x *= a
        df0_jitter.y *= a
        df0_jitter.z *= a
        data = df_jitter.to_numpy()
        jitterFileName = f'{os.getcwd()}/AOCS_amplitude{a}.txt'
        np.savetxt(jitterFileName, data, delimiter=' ', fmt=['%.3f', '%.9f', '%.9f', '%.9f'])
    
    for m,i in zip(magnitudes, range(len(df0))):       
        # No need to save the magnitude more than once
        df0[f'jitter{a}'].iloc[i] = np.array([nsr(a, m, model)])
        
df0['mag'] = np.array(magnitudes)
df0

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, column="ncon", passband='V', residuals="multi", 
                                show_ncam_noise_limits=1, show_saturation_limits=True,
                                show_targets=True,
                                legend=True, grid=False, figsize=(11,9))
# Set axes limits
for col, color in zip(cols[1:], colors):
    ax.plot(df0.mag, df0[col], 'o-', c=color)
ax.set_ylim(5, 1000)
ax.axvline(x=8.5, color="k", alpha=0.7, linestyle=':')  # Onset of saturation
ax.axvline(x=7.7, color="k", alpha=0.7, linestyle='-.') # When more than one pixel is saturated
ax.axvline(x=7.4, color="k", alpha=0.7, linestyle='--') # Blooming out of imagette
ax.set_title('a) Noise budget at camera level for BOL');

## Analysis of **Merged** N-CAM LCs (first sim)

In [ ]:
idir  = "/lhome/nicholas/data/platosimPaper/NSR_old" 
ofile = idir + "/results_per_star_old.ftr"

In [ ]:
# lcs = LightCurve(idir, mode="multi")
# lcs.run_NSRvsMag_analysis_perStar(ofile, 10000, suffix="hdf5")

In [ ]:
# Load results and sort logically
df = add_mag_column(vfile, pd.read_feather(ofile))
# Without contaminants
dc = df.loc[df.ncon == 0]

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, column="ncam", passband='V', residuals="multi", 
                                legend=True, grid=False, figsize=(9,6))
# Set axes limits
ax.set_title('b) Noise budget at instrument level for BOL')
ax.set_xlim(7, 12.77)
ax.set_ylim(5, 340)
# We change the fontsize of minor ticks label 
ticks_minor = [5, 6, 20, 30, 40, 50, 60, 70, 200, 300]
ax.yaxis.set_minor_locator(mticker.FixedLocator(ticks_minor))
ax.set_yticklabels(ticks_minor)
ticks_major = [1, 10, 100, 1000]
ax.yaxis.set_major_locator(mticker.FixedLocator(ticks_major))
ax.set_yticklabels(ticks_major);
# Save figure
fig.savefig('nsrInstrumentLevel.png', bbox_inches='tight', dpi=300);

In [ ]:
df.loc[(df.NSR <20) & (df.mag > 11)]

### Test nominal jitter vs. 5x the rms jitter

In [ ]:
# Load nominal jitter results
idir  = "/lhome/nicholas/data/platosimPaper/NSR_jitter1x" 
ofile = idir + "/results_per_star_jitter1x.ftr"

In [ ]:
# lcs = LightCurve(idir, mode="multi")
# lcs.run_NSRvsMag_analysis_perStar(ofile, 10000, suffix="hdf5")

In [ ]:
# Load results and sort logically
df1 = add_mag_column(vfile, pd.read_feather(ofile))
# Without contaminants
dc1 = df.loc[df.ncon == 0]

In [ ]:
# Load 5x jitter results
idir  = "/lhome/nicholas/data/platosimPaper/NSR_jitter5x" 
ofile = idir + "/results_per_star_jitter5x.ftr"

In [ ]:
# lcs = LightCurve(idir, mode="multi")
# lcs.run_NSRvsMag_analysis_perStar(ofile, 10000, suffix="hdf5")

In [ ]:
# Load results and sort logically
df5 = add_mag_column(vfile, pd.read_feather(ofile))
# Without contaminants
dc5 = df.loc[df.ncon == 0]

In [ ]:
# Calculate residuals
df['res'] = df5.NSR - df1.NSR
df

In [ ]:
# Plot all data together
fig, ax = pt.plotNSRvsMagnitude(df, column="ncam", passband='V', residuals='system', 
                                legend=True, grid=False, figsize=(11,8))